<a href="https://colab.research.google.com/github/BigBisus/Lab2-3Proch_Cache/blob/main/CacheLabs_lab2_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import threading
import time
from datetime import datetime, timedelta
from collections import defaultdict
import queue


class Cache:
    def __init__(self):
        # Словарь для хранения значений: {ключ: (значение, время окончания)}
        self._cache = {}
        # Блокировка для обеспечения потокобезопасности
        self._lock = threading.RLock()
        # Очередь для общения между основным кэшем и потоком очистки
        self._cleanup_queue = queue.Queue()
        # Запуск фонового потока для очистки устаревших ключей
        self._cleanup_thread = threading.Thread(target=self._cleanup_worker, daemon=True)
        self._cleanup_thread.start()

    def _is_expired(self, key):
        """Проверяет, истекло ли время жизни ключа"""
        with self._lock:
            if key not in self._cache:
                return True
            _, expiry_time = self._cache[key]
            return expiry_time is not None and datetime.now() > expiry_time

    def _cleanup_worker(self):
        """Фоновый поток для удаления устаревших ключей"""
        while True:
            try:
                # Проверяем, нужно ли завершить поток (например, через специальный сигнал)
                item = self._cleanup_queue.get(timeout=1)
                if item == "STOP":
                    break
            except queue.Empty:
                pass

            # Проходим по всем ключам и удаляем устаревшие
            with self._lock:
                expired_keys = []
                for key, (_, expiry_time) in self._cache.items():
                    if expiry_time is not None and datetime.now() > expiry_time:
                        expired_keys.append(key)

                for key in expired_keys:
                    del self._cache[key]

    def set(self, k, v, ttl=None):
        """
        Устанавливает значение в кэш
        :param k: ключ (не может быть None)
        :param v: значение
        :param ttl: время жизни в секундах (необязательно)
        """
        if k is None:
            raise ValueError("Key cannot be None")

        expiry_time = None
        if ttl is not None:
            expiry_time = datetime.now() + timedelta(seconds=ttl)

        with self._lock:
            self._cache[k] = (v, expiry_time)

    def get(self, k):
        """
        Получает значение из кэша
        :param k: ключ
        :return: значение или None, если ключ отсутствует или истек срок действия
        """
        if k is None:
            return None

        with self._lock:
            if k not in self._cache:
                return None

            value, expiry_time = self._cache[k]

            if expiry_time is not None and datetime.now() > expiry_time:
                # Удаляем устаревший ключ
                del self._cache[k]
                return None

            return value

    def delete(self, k):
        """
        Удаляет ключ из кэша
        :param k: ключ
        :return: True, если ключ был удален, иначе False
        """
        if k is None:
            return False

        with self._lock:
            if k in self._cache:
                del self._cache[k]
                return True
            return False

    def computeIfPresent(self, k, func):
        """
        Вычисляет новое значение, если ключ присутствует в кэше
        :param k: ключ
        :param func: функция, принимающая текущее значение и возвращающая новое
        :return: новое значение или None, если ключ отсутствует или истек
        """
        if k is None:
            return None

        with self._lock:
            if k not in self._cache:
                return None

            value, expiry_time = self._cache[k]

            if expiry_time is not None and datetime.now() > expiry_time:
                # Удаляем устаревший ключ
                del self._cache[k]
                return None

            # Вычисляем новое значение
            new_value = func(value)
            # Обновляем значение в кэше
            self._cache[k] = (new_value, expiry_time)
            return new_value

    def stop_cleanup_thread(self):
        """Метод для корректной остановки фонового потока (по желанию)"""
        self._cleanup_queue.put("STOP")


# Тестирование потокобезопасности метода computeIfPresent
def test_concurrent_compute_if_present():
    cache = Cache()
    # Инициализируем начальное значение
    cache.set("counter", 0)

    def increment_counter():
        for _ in range(10000):
            # Функция инкремента
            cache.computeIfPresent("counter", lambda x: x + 1)

    # Создаем три потока
    threads = []
    for i in range(3):
        thread = threading.Thread(target=increment_counter)
        threads.append(thread)
        thread.start()

    # Ждем завершения всех потоков
    for thread in threads:
        thread.join()

    # Проверяем результат
    final_value = cache.get("counter")
    print(f"Final counter value: {final_value}")




# Создаем кэш
cache = Cache()

# Устанавливаем значения
cache.set("key1", "value1")
cache.set("key2", "value2", ttl=2)  # TTL 2 секунды

# Получаем значения
print(cache.get("key1"))  # "value1"
print(cache.get("key2"))  # "value2"

# Ждем истечения времени
time.sleep(3)
print(cache.get("key1"))
print(cache.get("key2"))  # None (истекло)

# Удаляем ключ
print(cache.delete("key1"))  # True
print(cache.get("key1"))     # None

# Тест потокобезопасности
print("\nTesting concurrent computeIfPresent...")
test_concurrent_compute_if_present()

# Пример использования computeIfPresent
cache.set("number", 10)
result = cache.computeIfPresent("number", lambda x: x * 2)




value1
value2
value1
None
True
None

Testing concurrent computeIfPresent...
Final counter value: 30000


In [ ]:
import threading
import time
from datetime import datetime, timedelta


class Cache:
    def __init__(self):
        self._cache = {}
        self._lock = threading.Lock()

        # Поток для очистки устаревших ключей
        self._cleanup_thread = threading.Thread(target=self._cleanup_worker, daemon=True)
        self._cleanup_thread.start()

    def _cleanup_worker(self):
        while True:
            time.sleep(1)  # Проверяем раз в секунду
            now = datetime.now()
            with self._lock:
                expired = [k for k, (_, exp) in self._cache.items()
                          if exp and now > exp]
                for k in expired:
                    del self._cache[k]

    def set(self, k, v, ttl=None):
        if k is None: raise ValueError("Key cannot be None")
        expiry = datetime.now() + timedelta(seconds=ttl) if ttl else None
        with self._lock:
            self._cache[k] = (v, expiry)

    def get(self, k):
        if k is None: return None
        with self._lock:
            if k in self._cache:
                v, exp = self._cache[k]
                if exp and datetime.now() > exp:
                    del self._cache[k]
                    return None
                return v
        return None

    def delete(self, k):
        if k is None: return False
        with self._lock:
            return self._cache.pop(k, None) is not None

    def computeIfPresent(self, k, func):
        if k is None: return None
        with self._lock:
            if k in self._cache:
                v, exp = self._cache[k]
                if exp and datetime.now() > exp:
                    del self._cache[k]
                    return None
                new_v = func(v)
                self._cache[k] = (new_v, exp)
                return new_v
        return None


# Тест потокобезопасности
def test_concurrent():
    cache = Cache()
    cache.set("counter", 0)

    def worker(name):
        for i in range(10000):
            cache.computeIfPresent("counter", lambda x: x + 1)
            if i % 500 == 0:
                print(f"Thread {name}: {i}/10000")

    threads = [threading.Thread(target=worker, args=(i,)) for i in range(3)]
    for t in threads: t.start()
    for t in threads: t.join()

    print(f"Final: {cache.get('counter')}")


cache = Cache()
cache.set("test", 100, ttl=2)
print(cache.get("test"))
time.sleep(3)
print(cache.get("test"))  # None
test_concurrent()

100
None
Thread 0: 0/10000
Thread 1: 0/10000
Thread 2: 0/10000
Final: 300


In [ ]:
import threading
import time
from datetime import datetime, timedelta


class Cache:
    def __init__(self):
        self._cache = {}
        self._lock = threading.Lock()

        # Поток для очистки устаревших ключей
        self._cleanup_thread = threading.Thread(target=self._cleanup_worker, daemon=True)
        self._cleanup_thread.start()

    def _cleanup_worker(self):
        while True:
            time.sleep(1)  # Проверяем раз в секунду
            now = datetime.now()
            with self._lock:
              expired = [k for k, (_, exp) in self._cache.items()
                        if exp and now > exp]
              for k in expired:
                  del self._cache[k]

    def set(self, k, v, ttl=None):
        if k is None: raise ValueError("Key cannot be None")
        expiry = datetime.now() + timedelta(seconds=ttl) if ttl else None
        with self._lock:
          self._cache[k] = (v, expiry)

    def get(self, k):
        if k is None: return None
        with self._lock:
          if k in self._cache:
              v, exp = self._cache[k]
              if exp and datetime.now() > exp:
                  del self._cache[k]
                  return None
              return v
          return None

    def delete(self, k):
        if k is None: return False
        with self._lock:
          return self._cache.pop(k, None) is not None

    def computeIfPresent(self, k, func):
        if k is None: return None
        with self._lock:
          if k in self._cache:
              v, exp = self._cache[k]
              if exp and datetime.now() > exp:
                  del self._cache[k]
                  return None
              new_v = func(v)
              self._cache[k] = (new_v, exp)
              return new_v
          return None


# Тест потокобезопасности
def test_concurrent():
    cache = Cache()
    cache.set("counter", 0)

    def worker(name):
        for i in range(1000000):
            cache.computeIfPresent("counter", lambda x: x + 1)
            if i % 500 == 0:
                print(f"Thread {name}: {i}/10000")

    threads = [threading.Thread(target=worker, args=(i,)) for i in range(3)]
    for t in threads: t.start()
    for t in threads: t.join()

    final_value = cache.get("counter")
    print(f"Final value: {final_value}")



test_concurrent()

Выходные данные были обрезаны до нескольких последних строк (5000).
Thread 0: 168500/10000
Thread 0: 169000/10000
Thread 0: 169500/10000
Thread 2: 168000/10000
Thread 2: 168500/10000
Thread 0: 170000/10000
Thread 0: 170500/10000
Thread 1: 164000/10000
Thread 1: 164500/10000
Thread 2: 169000/10000
Thread 2: 169500/10000
Thread 0: 171000/10000
Thread 0: 171500/10000
Thread 1: 165000/10000
Thread 1: 165500/10000
Thread 2: 170000/10000
Thread 2: 170500/10000
Thread 0: 172000/10000
Thread 0: 172500/10000
Thread 0: 173000/10000
Thread 1: 166000/10000
Thread 1: 166500/10000
Thread 1: 167000/10000
Thread 0: 173500/10000
Thread 0: 174000/10000
Thread 2: 171000/10000
Thread 2: 171500/10000
Thread 1: 167500/10000
Thread 1: 168000/10000
Thread 1: 168500/10000
Thread 1: 169000/10000
Thread 1: 169500/10000
Thread 2: 172000/10000
Thread 2: 172500/10000
Thread 0: 174500/10000
Thread 0: 175000/10000
Thread 0: 175500/10000
Thread 2: 173000/10000
Thread 2: 173500/10000
Thread 1: 170000/10000
Thread 1: 17

In [ ]:
import threading
import time
from datetime import datetime, timedelta


class Cache:
    def __init__(self):
        self._cache = {}
        self._lock = threading.Lock()

        self._cleanup_thread = threading.Thread(target=self._cleanup_worker, daemon=True)
        self._cleanup_thread.start()

    def _cleanup_worker(self):
        while True:
            time.sleep(1)
            now = datetime.now()

            expired = [k for k, (_, exp) in self._cache.items()
                      if exp and now > exp]
            for k in expired:
                del self._cache[k]

    def set(self, k, v, ttl=None):
        if k is None: raise ValueError("Key cannot be None")
        expiry = datetime.now() + timedelta(seconds=ttl) if ttl else None

        self._cache[k] = (v, expiry)

    def get(self, k):
        if k is None: return None

        if k in self._cache:
            v, exp = self._cache[k]
            if exp and datetime.now() > exp:
                del self._cache[k]
                return None
            return v
        return None

    def delete(self, k):
        if k is None: return False

        return self._cache.pop(k, None) is not None

    def computeIfPresent(self, k, func):
        if k is None: return None
        if k in self._cache:
            v, exp = self._cache[k]
            if exp and datetime.now() > exp:
                del self._cache[k]
                return None
            new_v = func(v)
            self._cache[k] = (new_v, exp)
            return new_v
        return None


def test_concurrent():
    cache = Cache()
    cache.set("counter", 0)

    def worker(name):
        for i in range(100000):
            cache.computeIfPresent("counter", lambda x: x + 1)
            if i % 500 == 0:
                print(f"Thread {name}: {i}/10000")

    threads = [threading.Thread(target=worker, args=(i,)) for i in range(3)]
    for t in threads: t.start()
    for t in threads: t.join()

    return cache.get("counter")


print(f"Final value: {test_concurrent()}")


Thread 0: 0/10000
Thread 0: 500/10000
Thread 0: 1000/10000
Thread 0: 1500/10000
Thread 0: 2000/10000
Thread 0: 2500/10000
Thread 0: 3000/10000
Thread 0: 3500/10000
Thread 0: 4000/10000
Thread 0: 4500/10000
Thread 0: 5000/10000
Thread 0: 5500/10000
Thread 0: 6000/10000
Thread 0: 6500/10000
Thread 0: 7000/10000
Thread 0: 7500/10000
Thread 0: 8000/10000
Thread 0: 8500/10000
Thread 1: 0/10000
Thread 1: 500/10000
Thread 1: 1000/10000
Thread 1: 1500/10000
Thread 1: 2000/10000
Thread 1: 2500/10000
Thread 1: 3000/10000
Thread 1: 3500/10000
Thread 1: 4000/10000
Thread 1: 4500/10000
Thread 1: 5000/10000
Thread 1: 5500/10000
Thread 1: 6000/10000
Thread 0: 9000/10000
Thread 0: 9500/10000
Thread 0: 10000/10000
Thread 0: 10500/10000
Thread 0: 11000/10000
Thread 0: 11500/10000
Thread 0: 12000/10000
Thread 0: 12500/10000
Thread 0: 13000/10000
Thread 0: 13500/10000
Thread 0: 14000/10000
Thread 0: 14500/10000
Thread 0: 15000/10000
Thread 0: 15500/10000
Thread 0: 16000/10000
Thread 0: 16500/10000
Thread 

In [ ]:
import json
import os
import time
import threading

class WALCache:
    def __init__(self, wal_path):
        self.wal_path = wal_path
        self.cache = {}
        self.lock = threading.RLock()
        self._initialize_wal()
        self.compaction_thread = threading.Thread(target=self._compaction_worker, daemon=True)
        self.compaction_thread.start()

    def _initialize_wal(self):
        if os.path.exists(self.wal_path):
            with open(self.wal_path, 'r') as f:
                for line in f:
                    if line.strip():
                        entry = json.loads(line)
                        op = entry['operation']
                        k = entry['key']
                        if op == 'SET':
                            self.cache[k] = entry['value']
                        elif op == 'DELETE' and k in self.cache:
                            del self.cache[k]
        else:
            open(self.wal_path, 'w').close()

    def _append_to_wal(self, op, k, v=None):
        entry = {
            'timestamp': time.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3],  # короче, без микросекунд
            'operation': op,
            'key': k,
            'value': v
        }
        with open(self.wal_path, 'a') as f:
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')

    def set(self, k, v):
        with self.lock:
            self.cache[k] = v
            self._append_to_wal('SET', k, v)

    def get(self, k):
        with self.lock:
            return self.cache.get(k)

    def delete(self, k):
        with self.lock:
            if k in self.cache:
                del self.cache[k]
                self._append_to_wal('DELETE', k)

    def _compact_wal(self):
        temp = self.wal_path + '.tmp'
        with open(temp, 'w', encoding='utf-8') as f:
            for k, v in self.cache.items():
                entry = {
                    'timestamp': time.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3],
                    'operation': 'SET',
                    'key': k,
                    'value': v
                }
                f.write(json.dumps(entry, ensure_ascii=False) + '\n')
        os.replace(temp, self.wal_path)
        print("✅ Компактизация выполнена!")

    def _compaction_worker(self):
        while True:
            time.sleep(5)  # ⚠️ СДЕЛАЕМ 5 СЕКУНД ДЛЯ ТЕСТА!
            with self.lock:
                self._compact_wal()

# === ТЕСТ ===
wal_path = "/tmp/test_wal_debug.log"

# Удалим старый файл
if os.path.exists(wal_path):
    os.remove(wal_path)

print("Создаём кэш...")
cache = WALCache(wal_path)

print("Выполняем операции...")
cache.set("key1", "рош")
time.sleep(0.1)
cache.set("key2", "запрос")
time.sleep(0.1)
cache.delete("key1")
time.sleep(0.1)

print("Читаем WAL:")
with open(wal_path, 'r', encoding='utf-8') as f:
    lines = [l.strip() for l in f if l.strip()]
print(f"→ Записей: {len(lines)}")
for i, line in enumerate(lines, 1):
    print(f"{i}: {line}")
# Ждём 6 секунд — чтобы фоновый поток сработал (из-за time.sleep(5))
print("\nЖдём 6 секунд для компактизации...")
time.sleep(6)

print("После компактизации:")
with open(wal_path, 'r', encoding='utf-8') as f:
    lines = [l.strip() for l in f if l.strip()]
print(f"→ Записей: {len(lines)}")
for i, line in enumerate(lines, 1):
    print(f"{i}: {line}")


Создаём кэш...
Выполняем операции...
Читаем WAL:
→ Записей: 3
1: {"timestamp": "2026-06-06T08:26:13", "operation": "SET", "key": "key1", "value": "рош"}
2: {"timestamp": "2026-06-06T08:26:13", "operation": "SET", "key": "key2", "value": "запрос"}
3: {"timestamp": "2026-06-06T08:26:13", "operation": "DELETE", "key": "key1", "value": null}

Ждём 6 секунд для компактизации...
✅ Компактизация выполнена!
После компактизации:
→ Записей: 1
1: {"timestamp": "2026-06-06T08:26:18", "operation": "SET", "key": "key2", "value": "запрос"}


In [ ]:
# cache.py
class Cache:
    def __init__(self):
        self.data = {}

    def put(self, key, value):
        self.data[key] = value

    def get(self, key):
        return self.data.get(key)

    def items(self):
        return self.data.items()





In [ ]:
# sstable.py
import struct
from bisect import bisect_right

class SSTableWriter:
    HEADER_MAGIC = b"SST1"
    VERSION = 1

    def __init__(self, filename):
        self.filename = filename

    def write(self, cache, sparse_step=100):
        sorted_items = sorted(cache.items())
        sparse_index = []

        with open(self.filename, "wb") as f: # Исправлено: "wb " -> "wb"

            # HEADER
            f.write(self.HEADER_MAGIC)
            f.write(struct.pack("I", self.VERSION)) # Исправлено: "I " -> "I"

            data_start = f.tell()

            for i, (key, value) in enumerate(sorted_items):
                position = f.tell()

                if i % sparse_step == 0:
                     sparse_index.append((key, position))

                key_bytes = str(key).encode("utf-8") # Исправлено: "utf-8 " -> "utf-8"
                value_bytes = str(value).encode("utf-8") # Исправлено: "utf-8 " -> "utf-8"

                f.write(struct.pack("I", len(key_bytes))) # Исправлено: "I " -> "I"
                f.write(key_bytes)

                f.write(struct.pack("I", len(value_bytes))) # Исправлено: "I " -> "I"
                f.write(value_bytes)

            sparse_index_offset = f.tell()

            # SPARSE INDEX
            f.write(struct.pack("I", len(sparse_index))) # Исправлено: "I " -> "I"

            for key, position in sparse_index:
                key_bytes = str(key).encode("utf-8") # Исправлено: "utf-8 " -> "utf-8"

                f.write(struct.pack("I", len(key_bytes))) # Исправлено: "I " -> "I"
                f.write(key_bytes)

                f.write(struct.pack("Q", position)) # Исправлено: "Q " -> "Q"

            # FOOTER
            f.write(struct.pack("Q", sparse_index_offset)) # Исправлено: "Q " -> "Q"

        print(f"SSTable записана: {self.filename}")
        print(f"Записано элементов: {len(sorted_items)}")


class SSTableReader:
    HEADER_MAGIC = b"SST1" # Исправлено: b "SST1 " -> b"SST1"

    def __init__(self, filename):
        self.filename = filename
        self.index = []
        self._load_index()

    def _load_index(self):
        with open(self.filename, "rb") as f: # Исправлено: "rb " -> "rb"
            f.seek(-8, 2)
            sparse_index_offset = struct.unpack("Q", f.read(8))[0] # Исправлено: "Q " -> "Q"

            f.seek(sparse_index_offset)

            count = struct.unpack("I", f.read(4))[0] # Исправлено: "I " -> "I"

            for _ in range(count):
                key_len = struct.unpack("I", f.read(4))[0] # Исправлено: "I " -> "I"
                key = f.read(key_len).decode("utf-8") # Исправлено: "utf-8 " -> "utf-8"
                position = struct.unpack("Q", f.read(8))[0] # Исправлено: "Q " -> "Q"
                self.index.append((key, position))

    def get(self, search_key):
        search_key = str(search_key)
        keys = [k for k, _ in self.index]
        pos = bisect_right(keys, search_key) - 1

        if pos < 0:
            return None

        # Найдем начальную позицию в данных для поиска
        start_position = self.index[pos][1]
        # Определим конечную позицию (или используем конец файла)
        end_position = self.index[pos + 1][1] if pos + 1 < len(self.index) else None

        with open(self.filename, "rb") as f: # Исправлено: "rb " -> "rb"
            f.seek(start_position)

            while True:
                current_pos = f.tell()
                # Проверяем, не достигли ли мы границы следующего блока из индекса
                if end_position is not None and current_pos >= end_position:
                    break

                try:
                    key_len_data = f.read(4)
                    if not key_len_data:
                        break # Конец файла
                    key_len = struct.unpack("I", key_len_data)[0] # Исправлено: "I " -> "I"
                    key = f.read(key_len).decode("utf-8") # Исправлено: "utf-8 " -> "utf-8"

                    value_len_data = f.read(4)
                    if not value_len_data:
                         break # Конец файла
                    value_len = struct.unpack("I", value_len_data)[0] # Исправлено: "I " -> "I"
                    value = f.read(value_len).decode("utf-8") # Исправлено: "utf-8 " -> "utf-8"

                    if key == search_key:
                        return value
                    elif key > search_key: # Оптимизация: если текущий ключ больше искомого, можно выйти
                         break

                except Exception as e:
                    print(f"Ошибка при чтении: {e}")
                    break

        return None


In [ ]:
# main.py logic adapted for Colab
# Импорт уже выполнен, так как классы определены в текущем окружении

def create_cache():
    cache = Cache()
    for i in range(10000):
        cache.put(f"key{i:05d}", f"value{i}")
    return cache

def write_sstable():
    cache = create_cache()
    writer = SSTableWriter("data.sst")
    writer.write(cache) # Вызовет print внутри метода write
    # print("Файл SSTable создан") # Этот print теперь дублируется в методе write

def test_read():
    reader = SSTableReader("data.sst")
    test_keys = [
        "key00000",
        "key01234",
        "key05000",
        "key09999"
    ]

    print("Проверка чтения:")
    for key in test_keys:
        value = reader.get(key)
        print(f"GET({key}) -> {value}")

# Запуск основного процесса
write_sstable()
print("\n") # Пустая строка для разделения
test_read()

SSTable записана: data.sst
Записано элементов: 10000


Проверка чтения:
GET(key00000) -> value0
GET(key01234) -> value1234
GET(key05000) -> value5000
GET(key09999) -> value9999


In [ ]:
import time
import threading
import json
import os
import struct
from bisect import bisect_right, insort
from datetime import datetime, timedelta

# Подготовим тестовые данные
NUM_ENTRIES = 10000
test_data_dict = {f"key{i:05d}": f"value{i}" for i in range(NUM_ENTRIES)}
# Для тестов чтения по ключу
test_keys_to_read = ["key00000", "key01234", "key05000", "key09999"]
# Для тестов чтения всех записей
all_keys_to_read = list(test_data_dict.keys()) # Это будет список из 10,000 ключей

# --- LAB 2: Cache с TTL ---
class TTLCache:
    def __init__(self):
        self._cache = {}
        self._lock = threading.Lock()
        self._cleanup_thread = threading.Thread(target=self._cleanup_worker, daemon=True)
        self._cleanup_thread.start()

    def _cleanup_worker(self):
        while True:
            time.sleep(1)
            now = datetime.now()
            with self._lock:
                expired = [k for k, (_, exp) in self._cache.items() if exp and now > exp]
                for k in expired:
                    del self._cache[k]

    def set(self, k, v, ttl=None):
        if k is None: raise ValueError("Key cannot be None")
        expiry = datetime.now() + timedelta(seconds=ttl) if ttl else None
        with self._lock:
            self._cache[k] = (v, expiry)

    def get(self, k):
        if k is None: return None
        with self._lock:
            if k in self._cache:
                v, exp = self._cache[k]
                if exp and datetime.now() > exp:
                    del self._cache[k]
                    return None
                return v
        return None

def benchmark_lab2():
    print("--- Тестирование Lab 2: Cache с TTL (10,000 записей) ---")
    cache = TTLCache()

    # Измерение времени записи
    start_write = time.perf_counter()
    for k, v in test_data_dict.items():
        cache.set(k, v)
    end_write = time.perf_counter()
    write_time = end_write - start_write
    print(f"Lab 2 - Время записи 10,000 записей: {write_time:.4f} секунд")

    # Измерение времени чтения всех 10,000 записей
    start_read = time.perf_counter()
    for key in all_keys_to_read:
        val = cache.get(key)
        # print(f"Lab2 GET({key}) -> {val}") # Не будем выводить, чтобы не засорять вывод при замерах
    end_read = time.perf_counter()
    read_time = end_read - start_read
    print(f"Lab 2 - Время чтения 10,000 записей: {read_time:.4f} секунд")
    print("-" * 40)
    return write_time, read_time


# --- LAB 3: WAL Cache ---
class WALCache:
    def __init__(self, wal_path):
        self.wal_path = wal_path
        self.cache = {}
        self.lock = threading.RLock()
        self._initialize_wal()
        # Отключим фоновую компактизацию для чистоты эксперимента
        # self.compaction_thread = threading.Thread(target=self._compaction_worker, daemon=True)
        # self.compaction_thread.start()

    def _initialize_wal(self):
        if os.path.exists(self.wal_path):
            with open(self.wal_path, 'r') as f:
                for line in f:
                    if line.strip():
                        entry = json.loads(line)
                        op = entry['operation']
                        k = entry['key']
                        if op == 'SET':
                            self.cache[k] = entry['value']
                        elif op == 'DELETE' and k in self.cache:
                            del self.cache[k]
        else:
            open(self.wal_path, 'w').close()

    def _append_to_wal(self, op, k, v=None):
        entry = {
            'timestamp': time.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3],
            'operation': op,
            'key': k,
            'value': v
        }
        with open(self.wal_path, 'a') as f:
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')

    def set(self, k, v):
        with self.lock:
            self.cache[k] = v
            self._append_to_wal('SET', k, v)

    def get(self, k):
        with self.lock:
            return self.cache.get(k)

    def delete(self, k):
        with self.lock:
            if k in self.cache:
                del self.cache[k]
                self._append_to_wal('DELETE', k)

def benchmark_lab3():
    print("--- Тестирование Lab 3: WAL Cache (10,000 записей) ---")
    wal_path = "temp_wal_test_benchmark.log"
    # Удалим старый файл WAL, если он есть
    if os.path.exists(wal_path):
        os.remove(wal_path)

    cache = WALCache(wal_path)

    # Измерение времени записи
    start_write = time.perf_counter()
    for k, v in test_data_dict.items():
        cache.set(k, v)
    end_write = time.perf_counter()
    write_time = end_write - start_write
    print(f"Lab 3 - Время записи 10,000 записей: {write_time:.4f} секунд")

    # Измерение времени чтения всех 10,000 записей
    start_read = time.perf_counter()
    for key in all_keys_to_read:
        val = cache.get(key)
        # print(f"Lab3 GET({key}) -> {val}") # Не будем выводить, чтобы не засорять вывод при замерах
    end_read = time.perf_counter()
    read_time = end_read - start_read
    print(f"Lab 3 - Время чтения 10,000 записей: {read_time:.4f} секунд")

    # Удалим файл WAL после теста
    if os.path.exists(wal_path):
        os.remove(wal_path)

    print("-" * 40)
    return write_time, read_time


# --- LAB 4: SSTable ---
class CacheForSSTable:
    def __init__(self):
        self.data = {}

    def put(self, key, value):
        self.data[key] = value

    def get(self, key):
        return self.data.get(key)

    def items(self):
        return self.data.items()

# ИСПРАВЛЕННЫЙ КЛАСС SSTableWriter (см. предыдущие комментарии о пробелах)
class SSTableWriterBenchmark:
    HEADER_MAGIC = b"SST1"
    VERSION = 1

    def __init__(self, filename):
        self.filename = filename

    def write(self, cache, sparse_step=100):
        sorted_items = sorted(cache.items())
        sparse_index = []

        # ИСПРАВЛЕНО: Убраны лишние пробелы
        with open(self.filename, "wb") as f: # было "wb "
            f.write(self.HEADER_MAGIC)
            f.write(struct.pack("I", self.VERSION)) # было "I "

            for i, (key, value) in enumerate(sorted_items):
                position = f.tell()
                if i % sparse_step == 0:
                     sparse_index.append((key, position))
                key_bytes = str(key).encode("utf-8") # было "utf-8 "
                value_bytes = str(value).encode("utf-8") # было "utf-8 "

                f.write(struct.pack("I", len(key_bytes))) # было "I "
                f.write(key_bytes)
                f.write(struct.pack("I", len(value_bytes))) # было "I "
                f.write(value_bytes)

            sparse_index_offset = f.tell()
            f.write(struct.pack("I", len(sparse_index))) # было "I "

            for key, position in sparse_index:
                key_bytes = str(key).encode("utf-8") # было "utf-8 "
                f.write(struct.pack("I", len(key_bytes))) # было "I "
                f.write(key_bytes)
                f.write(struct.pack("Q", position)) # было "Q "

            f.write(struct.pack("Q", sparse_index_offset)) # было "Q "

        # print(f"SSTable записана: {self.filename}") # Убираем для чистоты
        # print(f"Записано элементов: {len(sorted_items)}") # Убираем для чистоты

# ИСПРАВЛЕННЫЙ КЛАСС SSTableReader
class SSTableReaderBenchmark:
    HEADER_MAGIC = b"SST1" # Исправлено: b "SST1 " -> b"SST1"

    def __init__(self, filename):
        self.filename = filename
        self.index = []
        self.sorted_full_keys = [] # Добавим список всех ключей для итерации
        self._load_index_and_keys()

    def _load_index_and_keys(self):
        with open(self.filename, "rb") as f: # Исправлено: "rb " -> "rb"
            f.seek(-8, 2)
            sparse_index_offset = struct.unpack("Q", f.read(8))[0] # Исправлено: "Q " -> "Q"

            # Загрузка индекса (как раньше)
            f.seek(sparse_index_offset)
            count = struct.unpack("I", f.read(4))[0] # Исправлено: "I " -> "I"
            for _ in range(count):
                key_len = struct.unpack("I", f.read(4))[0] # Исправлено: "I " -> "I"
                key = f.read(key_len).decode("utf-8") # Исправлено: "utf-8 " -> "utf-8"
                position = struct.unpack("Q", f.read(8))[0] # Исправлено: "Q " -> "Q"
                self.index.append((key, position))

            # Сортировка индекса (может быть уже отсортирован, но на всякий случай)
            self.index.sort(key=lambda x: x[0])

            # --- НОВОЕ: Загрузка всех ключей и их позиций для итерации ---
            f.seek(len(self.HEADER_MAGIC) + 4) # Перейти к началу блока данных
            all_keys_positions = []
            while f.tell() < sparse_index_offset:
                 try:
                    key_len_data = f.read(4)
                    if not key_len_data:
                        break
                    key_len = struct.unpack("I", key_len_data)[0] # Исправлено: "I " -> "I"
                    key = f.read(key_len).decode("utf-8") # Исправлено: "utf-8 " -> "utf-8"
                    value_len_data = f.read(4)
                    if not value_len_data:
                        break
                    value_len = struct.unpack("I", value_len_data)[0] # Исправлено: "I " -> "I"
                    # Пропускаем значение
                    f.read(value_len)

                    all_keys_positions.append((key, f.tell())) # Сохраняем ключ и позицию после него

                 except Exception as e:
                     print(f"Ошибка при чтении данных для итерации: {e}")
                     break

            # Сортируем все ключи
            all_keys_positions.sort(key=lambda x: x[0])
            self.sorted_full_keys = [item[0] for item in all_keys_positions]


    def get(self, search_key):
        search_key = str(search_key)
        keys = [k for k, _ in self.index]
        pos = bisect_right(keys, search_key) - 1

        if pos < 0:
            return None

        start_position = self.index[pos][1]
        end_position = self.index[pos + 1][1] if pos + 1 < len(self.index) else None

        with open(self.filename, "rb") as f: # Исправлено: "rb " -> "rb"
            f.seek(start_position)

            while True:
                current_pos = f.tell()
                if end_position is not None and current_pos >= end_position:
                    break

                try:
                    key_len_data = f.read(4)
                    if not key_len_data:
                        break
                    key_len = struct.unpack("I", key_len_data)[0] # Исправлено: "I " -> "I"
                    key = f.read(key_len).decode("utf-8") # Исправлено: "utf-8 " -> "utf-8"

                    value_len_data = f.read(4)
                    if not value_len_data:
                         break
                    value_len = struct.unpack("I", value_len_data)[0] # Исправлено: "I " -> "I"
                    value = f.read(value_len).decode("utf-8") # Исправлено: "utf-8 " -> "utf-8"

                    if key == search_key:
                        return value
                    elif key > search_key:
                         break

                except Exception as e:
                    # print(f"Ошибка при чтении: {e}") # Убираем для чистоты
                    break

        return None

    # Новый метод для получения значения по индексу в отсортированном списке
    def get_by_index(self, idx):
        if 0 <= idx < len(self.sorted_full_keys):
             key = self.sorted_full_keys[idx]
             # Используем обычный get для поиска значения по ключу
             return self.get(key)
        return None


def benchmark_lab4():
    print("--- Тестирование Lab 4: SSTable (10,000 записей) ---")
    sstable_file = "temp_sstable_test_benchmark.sst"

    # Подготовим кэш с данными
    cache_for_sstable = CacheForSSTable()
    for k, v in test_data_dict.items():
        cache_for_sstable.put(k, v)

    # Создадим writer и измерим время записи
    writer = SSTableWriterBenchmark(sstable_file)
    start_write = time.perf_counter()
    writer.write(cache_for_sstable)
    end_write = time.perf_counter()
    write_time = end_write - start_write
    print(f"Lab 4 - Время записи 10,000 записей в SSTable: {write_time:.4f} секунд")

    # Создадим reader
    reader = SSTableReaderBenchmark(sstable_file)
    # Проверим, что количество загруженных ключей совпадает
    assert len(reader.sorted_full_keys) == NUM_ENTRIES, f"Количество загруженных ключей {len(reader.sorted_full_keys)} не равно {NUM_ENTRIES}"

    # Измерение времени чтения всех 10,000 записей через индекс (медленно!)
    start_read = time.perf_counter()
    for i in range(len(all_keys_to_read)): # Используем длину списка для итерации по индексам
        val = reader.get_by_index(i)
        # print(f"Lab4 GET_BY_IDX({i}, key={reader.sorted_full_keys[i]}) -> {val}") # Не будем выводить
    end_read = time.perf_counter()
    read_time = end_read - start_read
    print(f"Lab 4 - Время чтения 10,000 записей (итерация по индексу): {read_time:.4f} секунд")

    # Удалим файл SSTable после теста
    if os.path.exists(sstable_file):
        os.remove(sstable_file)

    print("-" * 40)
    return write_time, read_time


# --- Запуск тестов ---
if __name__ == "__main__":
    print(f"Запуск тестов с {NUM_ENTRIES} записями...\n")
    lab2_times = benchmark_lab2()
    lab3_times = benchmark_lab3()
    lab4_times = benchmark_lab4()

    print("\n=== Сводка по времени (10,000 записей) ===")
    print(f"Lab 2 (TTL Cache) - Запись: {lab2_times[0]:.4f}s, Чтение: {lab2_times[1]:.4f}s")
    print(f"Lab 3 (WAL Cache) - Запись: {lab3_times[0]:.4f}s, Чтение: {lab3_times[1]:.4f}s")
    print(f"Lab 4 (SSTable)   - Запись: {lab4_times[0]:.4f}s, Чтение (итерация): {lab4_times[1]:.4f}s")



Запуск тестов с 10000 записями...

--- Тестирование Lab 2: Cache с TTL (10,000 записей) ---
Lab 2 - Время записи 10,000 записей: 0.0151 секунд
Lab 2 - Время чтения 10,000 записей: 0.0100 секунд
----------------------------------------
--- Тестирование Lab 3: WAL Cache (10,000 записей) ---
Lab 3 - Время записи 10,000 записей: 0.4181 секунд
Lab 3 - Время чтения 10,000 записей: 0.0087 секунд
----------------------------------------
--- Тестирование Lab 4: SSTable (10,000 записей) ---
Lab 4 - Время записи 10,000 записей в SSTable: 0.0385 секунд
Lab 4 - Время чтения 10,000 записей (итерация по индексу): 0.9645 секунд
----------------------------------------

=== Сводка по времени (10,000 записей) ===
Lab 2 (TTL Cache) - Запись: 0.0151s, Чтение: 0.0100s
Lab 3 (WAL Cache) - Запись: 0.4181s, Чтение: 0.0087s
Lab 4 (SSTable)   - Запись: 0.0385s, Чтение (итерация): 0.9645s


In [ ]:
import time
import threading
import json
import os
import struct
from bisect import bisect_right
from datetime import datetime, timedelta

# Подготовим тестовые данные
NUM_ENTRIES = 10000
test_data = {f"key{i:05d}": f"value{i}" for i in range(NUM_ENTRIES)}
test_keys_to_read = ["key00000", "key01234", "key05000", "key09999"]

# --- LAB 2: Cache с TTL ---
class TTLCache:
    def __init__(self):
        self._cache = {}
        self._lock = threading.Lock()
        self._cleanup_thread = threading.Thread(target=self._cleanup_worker, daemon=True)
        self._cleanup_thread.start()

    def _cleanup_worker(self):
        while True:
            time.sleep(1)
            now = datetime.now()
            with self._lock:
                expired = [k for k, (_, exp) in self._cache.items() if exp and now > exp]
                for k in expired:
                    del self._cache[k]

    def set(self, k, v, ttl=None):
        if k is None: raise ValueError("Key cannot be None")
        expiry = datetime.now() + timedelta(seconds=ttl) if ttl else None
        with self._lock:
            self._cache[k] = (v, expiry)

    def get(self, k):
        if k is None: return None
        with self._lock:
            if k in self._cache:
                v, exp = self._cache[k]
                if exp and datetime.now() > exp:
                    del self._cache[k]
                    return None
                return v
        return None

def benchmark_lab2():
    print("--- Тестирование Lab 2: Cache с TTL ---")
    cache = TTLCache()

    # Измерение времени записи
    start_write = time.perf_counter()
    for k, v in test_data.items():
        cache.set(k, v)
    end_write = time.perf_counter()
    write_time = end_write - start_write
    print(f"Lab 2 - Время записи 10,000 записей: {write_time:.4f} секунд")

    # Измерение времени чтения
    start_read = time.perf_counter()
    for key in test_keys_to_read:
        val = cache.get(key)
        # print(f"Lab2 GET({key}) -> {val}") # Не будем выводить, чтобы не засорять вывод при замерах
    end_read = time.perf_counter()
    read_time = end_read - start_read
    print(f"Lab 2 - Время чтения 4 записей: {read_time:.6f} секунд")
    print("-" * 40)
    return write_time, read_time


# --- LAB 3: WAL Cache ---
class WALCache:
    def __init__(self, wal_path):
        self.wal_path = wal_path
        self.cache = {}
        self.lock = threading.RLock()
        self._initialize_wal()
        # Отключим фоновую компактизацию для чистоты эксперимента
        # self.compaction_thread = threading.Thread(target=self._compaction_worker, daemon=True)
        # self.compaction_thread.start()

    def _initialize_wal(self):
        if os.path.exists(self.wal_path):
            with open(self.wal_path, 'r') as f:
                for line in f:
                    if line.strip():
                        entry = json.loads(line)
                        op = entry['operation']
                        k = entry['key']
                        if op == 'SET':
                            self.cache[k] = entry['value']
                        elif op == 'DELETE' and k in self.cache:
                            del self.cache[k]
        else:
            open(self.wal_path, 'w').close()

    def _append_to_wal(self, op, k, v=None):
        entry = {
            'timestamp': time.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3],
            'operation': op,
            'key': k,
            'value': v
        }
        with open(self.wal_path, 'a') as f:
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')

    def set(self, k, v):
        with self.lock:
            self.cache[k] = v
            self._append_to_wal('SET', k, v)

    def get(self, k):
        with self.lock:
            return self.cache.get(k)

    def delete(self, k):
        with self.lock:
            if k in self.cache:
                del self.cache[k]
                self._append_to_wal('DELETE', k)

    # Отключаем фоновую компактизацию для чистоты измерений
    # def _compaction_worker(self):
    #     while True:
    #         time.sleep(5)
    #         with self.lock:
    #             self._compact_wal()

def benchmark_lab3():
    print("--- Тестирование Lab 3: WAL Cache ---")
    wal_path = "temp_wal_test.log"
    # Удалим старый файл WAL, если он есть
    if os.path.exists(wal_path):
        os.remove(wal_path)

    cache = WALCache(wal_path)

    # Измерение времени записи
    start_write = time.perf_counter()
    for k, v in test_data.items():
        cache.set(k, v)
    end_write = time.perf_counter()
    write_time = end_write - start_write
    print(f"Lab 3 - Время записи 10,000 записей: {write_time:.4f} секунд")

    # Измерение времени чтения
    start_read = time.perf_counter()
    for key in test_keys_to_read:
        val = cache.get(key)
        # print(f"Lab3 GET({key}) -> {val}") # Не будем выводить, чтобы не засорять вывод при замерах
    end_read = time.perf_counter()
    read_time = end_read - start_read
    print(f"Lab 3 - Время чтения 4 записей: {read_time:.6f} секунд")

    # Удалим файл WAL после теста
    if os.path.exists(wal_path):
        os.remove(wal_path)

    print("-" * 40)
    return write_time, read_time


# --- LAB 4: SSTable ---
class CacheForSSTable: # Используем для Lab 4, чтобы не путаться с TTLCache
    def __init__(self):
        self.data = {}

    def put(self, key, value):
        self.data[key] = value

    def get(self, key):
        return self.data.get(key)

    def items(self):
        return self.data.items()

class SSTableWriterBenchmark:
    HEADER_MAGIC = b"SST1"
    VERSION = 1

    def __init__(self, filename):
        self.filename = filename

    def write(self, cache, sparse_step=100):
        sorted_items = sorted(cache.items())
        sparse_index = []

        # ИСПРАВЛЕНО: Убраны лишние пробелы
        with open(self.filename, "wb") as f: # было "wb "
            f.write(self.HEADER_MAGIC)
            f.write(struct.pack("I", self.VERSION)) # было "I "

            for i, (key, value) in enumerate(sorted_items):
                position = f.tell()
                if i % sparse_step == 0:
                     sparse_index.append((key, position))
                key_bytes = str(key).encode("utf-8") # было "utf-8 "
                value_bytes = str(value).encode("utf-8") # было "utf-8 "

                f.write(struct.pack("I", len(key_bytes))) # было "I "
                f.write(key_bytes)
                f.write(struct.pack("I", len(value_bytes))) # было "I "
                f.write(value_bytes)

            sparse_index_offset = f.tell()
            f.write(struct.pack("I", len(sparse_index))) # было "I "

            for key, position in sparse_index:
                key_bytes = str(key).encode("utf-8") # было "utf-8 "
                f.write(struct.pack("I", len(key_bytes))) # было "I "
                f.write(key_bytes)
                f.write(struct.pack("Q", position)) # было "Q "

            f.write(struct.pack("Q", sparse_index_offset)) # было "Q "

        # print(f"SSTable записана: {self.filename}") # Убираем для чистоты
        # print(f"Записано элементов: {len(sorted_items)}") # Убираем для чистоты

class SSTableReaderBenchmark:
    HEADER_MAGIC = b"SST1" # Исправлено: b "SST1 " -> b"SST1"

    def __init__(self, filename):
        self.filename = filename
        self.index = []
        self._load_index()

    def _load_index(self):
        with open(self.filename, "rb") as f: # Исправлено: "rb " -> "rb"
            f.seek(-8, 2)
            sparse_index_offset = struct.unpack("Q", f.read(8))[0] # Исправлено: "Q " -> "Q"

            f.seek(sparse_index_offset)

            count = struct.unpack("I", f.read(4))[0] # Исправлено: "I " -> "I"

            for _ in range(count):
                key_len = struct.unpack("I", f.read(4))[0] # Исправлено: "I " -> "I"
                key = f.read(key_len).decode("utf-8") # Исправлено: "utf-8 " -> "utf-8"
                position = struct.unpack("Q", f.read(8))[0] # Исправлено: "Q " -> "Q"
                self.index.append((key, position))

    def get(self, search_key):
        search_key = str(search_key)
        keys = [k for k, _ in self.index]
        pos = bisect_right(keys, search_key) - 1

        if pos < 0:
            return None

        # Найдем начальную позицию в данных для поиска
        start_position = self.index[pos][1]
        # Определим конечную позицию (или используем конец файла)
        end_position = self.index[pos + 1][1] if pos + 1 < len(self.index) else None

        with open(self.filename, "rb") as f: # Исправлено: "rb " -> "rb"
            f.seek(start_position)

            while True:
                current_pos = f.tell()
                # Проверяем, не достигли ли мы границы следующего блока из индекса
                if end_position is not None and current_pos >= end_position:
                    break

                try:
                    key_len_data = f.read(4)
                    if not key_len_data:
                        break # Конец файла
                    key_len = struct.unpack("I", key_len_data)[0] # Исправлено: "I " -> "I"
                    key = f.read(key_len).decode("utf-8") # Исправлено: "utf-8 " -> "utf-8"

                    value_len_data = f.read(4)
                    if not value_len_data:
                         break # Конец файла
                    value_len = struct.unpack("I", value_len_data)[0] # Исправлено: "I " -> "I"
                    value = f.read(value_len).decode("utf-8") # Исправлено: "utf-8 " -> "utf-8"

                    if key == search_key:
                        return value
                    elif key > search_key: # Оптимизация: если текущий ключ больше искомого, можно выйти
                         break

                except Exception as e:
                    # print(f"Ошибка при чтении: {e}") # Убираем для чистоты
                    break

        return None

def benchmark_lab4():
    print("--- Тестирование Lab 4: SSTable ---")
    sstable_file = "temp_sstable_test.sst"

    # Подготовим кэш с данными
    cache_for_sstable = CacheForSSTable()
    for k, v in test_data.items():
        cache_for_sstable.put(k, v)

    # Создадим writer и измерим время записи
    writer = SSTableWriterBenchmark(sstable_file)
    start_write = time.perf_counter()
    writer.write(cache_for_sstable)
    end_write = time.perf_counter()
    write_time = end_write - start_write
    print(f"Lab 4 - Время записи 10,000 записей в SSTable: {write_time:.4f} секунд")

    # Создадим reader и измерим время чтения
    reader = SSTableReaderBenchmark(sstable_file)
    start_read = time.perf_counter()
    for key in test_keys_to_read:
        val = reader.get(key)
        # print(f"Lab4 GET({key}) -> {val}") # Не будем выводить, чтобы не засорять вывод при замерах
    end_read = time.perf_counter()
    read_time = end_read - start_read
    print(f"Lab 4 - Время чтения 4 записей из SSTable: {read_time:.6f} секунд")

    # Удалим файл SSTable после теста
    if os.path.exists(sstable_file):
        os.remove(sstable_file)

    print("-" * 40)
    return write_time, read_time


# --- Запуск тестов ---
if __name__ == "__main__":
    lab2_times = benchmark_lab2()
    lab3_times = benchmark_lab3()
    lab4_times = benchmark_lab4()

    print("\n=== Сводка по времени ===")
    print(f"Lab 2 (TTL Cache) - Запись: {lab2_times[0]:.4f}s, Чтение: {lab2_times[1]:.6f}s")
    print(f"Lab 3 (WAL Cache) - Запись: {lab3_times[0]:.4f}s, Чтение: {lab3_times[1]:.6f}s")
    print(f"Lab 4 (SSTable)   - Запись: {lab4_times[0]:.4f}s, Чтение: {lab4_times[1]:.6f}s")


--- Тестирование Lab 2: Cache с TTL ---
Lab 2 - Время записи 10,000 записей: 0.0064 секунд
Lab 2 - Время чтения 4 записей: 0.000010 секунд
----------------------------------------
--- Тестирование Lab 3: WAL Cache ---
Lab 3 - Время записи 10,000 записей: 0.3687 секунд
Lab 3 - Время чтения 4 записей: 0.000015 секунд
----------------------------------------
--- Тестирование Lab 4: SSTable ---
Lab 4 - Время записи 10,000 записей в SSTable: 0.0525 секунд
Lab 4 - Время чтения 4 записей из SSTable: 0.000509 секунд
----------------------------------------

=== Сводка по времени ===
Lab 2 (TTL Cache) - Запись: 0.0064s, Чтение: 0.000010s
Lab 3 (WAL Cache) - Запись: 0.3687s, Чтение: 0.000015s
Lab 4 (SSTable)   - Запись: 0.0525s, Чтение: 0.000509s
